# 🤖 AI-Powered Python Training Chatbot
### NTUC LearningHub — Mini Project 3
**Skills:** OpenAI API · Prompt Engineering · Conversation Memory · Graceful Fallback  
**Use Case:** A chatbot embedded in the course portal to answer student Python questions 24/7.
---


## 1️⃣ Setup & Configuration

In [ ]:
import os

try:
    import openai
    OPENAI_OK = True
    print("✅ openai library available")
except ImportError:
    OPENAI_OK = False
    print("ℹ  pip install openai  (running with fallback responses)")

API_KEY = os.environ.get("OPENAI_API_KEY", "")
USE_LIVE = bool(API_KEY) and OPENAI_OK
print(f"Mode: {'🟢 Live GPT-4o' if USE_LIVE else '🟡 Demo Fallback (set OPENAI_API_KEY to enable live)'}")

## 2️⃣ System Prompt Design
Good system prompts are the foundation of a well-behaved AI assistant.

In [ ]:
SYSTEM_PROMPT = """
You are PythonBot, a friendly and expert Python programming assistant
for students at NTUC LearningHub. Your role is to:

1. Explain Python concepts clearly with practical, real-world examples.
2. Debug student code patiently and explain the fix, not just provide it.
3. Suggest exercises to reinforce learning.
4. Use Singaporean context in examples when relevant (HDB, MRT, CPF, etc.).
5. Encourage learners and celebrate their progress.
6. Keep responses concise — under 200 words unless the topic demands more.

You support these NTUC LearningHub Python courses:
- Fundamentals of Python Programming
- Analyse Business Data Using Python
- Advanced Analytics and ML Using Python
- Deep Learning Models and AI Using Python
"""
print("System prompt ready — character count:", len(SYSTEM_PROMPT))

## 3️⃣ Chatbot Class

In [ ]:
FALLBACKS = {
    "list": (
        "A Python list is an ordered, mutable collection.\n\n"
        "  mrt_stations = ['Jurong East', 'Buona Vista', 'Orchard']\n"
        "  print(mrt_stations[0])  # Jurong East\n\n"
        "Lists support indexing, slicing, append(), remove(), and len()."
    ),
    "loop": (
        "Python has two main loops:\n\n"
        "  for i in range(5): print(i)     # 0 to 4\n\n"
        "  count = 0\n"
        "  while count < 5: count += 1     # until False\n\n"
        "Use 'for' when count is known; 'while' when it isn't."
    ),
    "function": (
        "Functions are reusable blocks:\n\n"
        "  def calculate_cpf(salary, rate=0.20):\n"
        "      return salary * rate\n\n"
        "  print(calculate_cpf(5000))  # 1000.0"
    ),
    "pandas": (
        "pandas is the core data library:\n\n"
        "  import pandas as pd\n"
        "  df = pd.read_csv('data.csv')\n"
        "  df.head()          # first 5 rows\n"
        "  df['Salary'].mean()  # average salary"
    ),
    "class": (
        "Classes define objects with attributes and methods:\n\n"
        "  class Trainer:\n"
        "      def __init__(self, name, domain):\n"
        "          self.name = name\n"
        "          self.domain = domain\n\n"
        "      def introduce(self):\n"
        "          print(f'Hi, I am {self.name}, teaching {self.domain}')"
    ),
}

class PythonBot:
    def __init__(self):
        self.history = []
        if USE_LIVE:
            self.client = openai.OpenAI(api_key=API_KEY)

    def chat(self, msg):
        self.history.append({"role":"user","content":msg})
        reply = self._live(msg) if USE_LIVE else self._fallback(msg)
        self.history.append({"role":"assistant","content":reply})
        return reply

    def _live(self, msg):
        try:
            r = self.client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role":"system","content":SYSTEM_PROMPT}] + self.history[-10:],
                max_tokens=400, temperature=0.6)
            return r.choices[0].message.content
        except Exception as e:
            return f"[API Error: {e}]\n\n{self._fallback(msg)}"

    def _fallback(self, msg):
        ml = msg.lower()
        for kw, resp in FALLBACKS.items():
            if kw in ml: return resp
        return ("Great question! Set OPENAI_API_KEY for a live GPT-4o answer.\n"
                "Try asking about: list, loop, function, pandas, or class.")

    def reset(self):
        self.history = []
        print("🔄 Conversation reset")

bot = PythonBot()
print("✅ PythonBot initialised")

## 4️⃣ Demo Conversation

In [ ]:
demo_questions = [
    "What is a Python list and how do I use it?",
    "How do I read a CSV file using pandas?",
    "Can you show me how to write a Python class?",
]

for q in demo_questions:
    print(f"\n{'─'*60}")
    print(f"🎓 Student : {q}")
    print(f"\n🤖 PythonBot:\n{bot.chat(q)}")
print(f"\n{'─'*60}")

## 5️⃣ Prompt Engineering Tips

In [ ]:
tips = {
    "Be specific"       : "Instead of 'explain Python', ask 'explain Python list comprehension with an example'",
    "Provide context"   : "Tell the model the student level: 'explain recursion to a beginner'",
    "Ask for format"    : "Ask for code + explanation: 'show the code and then explain each line'",
    "Use role framing"  : "SYSTEM prompt sets behaviour — this is the most powerful lever",
    "Limit scope"       : "max_tokens controls length; temperature 0.2–0.4 for factual tech answers",
}

print("═"*55)
print("  PROMPT ENGINEERING BEST PRACTICES")
print("═"*55)
for tip, detail in tips.items():
    print(f"\n  ✅ {tip}")
    print(f"     {detail}")